# SixDegrees — Pipeline Demo

Demonstrates two convergence scenarios using the live backend pipeline (no Supabase calls — all in-memory).

- **Case 1:** Eleanor and Brita have identical profiles but their interaction intensity increases across 6 frames.
- **Case 2:** Eleanor and Brita have zero interaction but Brita's profile converges toward Eleanor's across 6 frames.

In [ ]:
import os
import sys

# Inject dummy Supabase env vars so config/settings.py can be imported
# without a live Supabase connection.  Must happen before any backend import.
os.environ.setdefault('SUPABASE_URL', 'http://localhost')
os.environ.setdefault('SUPABASE_KEY', 'demo-key')

# Make backend packages importable
sys.path.insert(0, '../backend')

import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — works in headless / CI environments
import matplotlib.pyplot as plt

from models.user import UserProfile
from services.map.contracts import PipelineInput
from services.map.distance import build_combined_distance
from services.map.projector import project

print('Imports OK')

In [ ]:
# ---------------------------------------------------------------------------
# Core users
# ---------------------------------------------------------------------------

ELEANOR = UserProfile(
    id='eleanor',
    nickname='Eleanor',
    city='NYC',
    state='NY',
    age=28,
    interests=['art', 'music', 'film'],
    education='bachelor',
    industry='media',
    languages=['english'],
)

BRITA_ORIGINAL = UserProfile(
    id='brita',
    nickname='Brita',
    city='Austin',
    state='TX',
    age=45,
    interests=['sports', 'finance', 'cooking'],
    education='phd',
    industry='tech',
    languages=['german', 'spanish'],
)

# 5 close friends of Eleanor — heavy interaction weight
# raw_weight = 30*0.3 + 20*0.5 + 10*0.2 = 9.0 + 10.0 + 2.0 = 21.0
FRIENDS = [
    UserProfile(id='friend_01', nickname='friend_01', city='NYC', state='NY', age=27,
                interests=['art', 'music'], education='bachelor', industry='media',
                languages=['english']),
    UserProfile(id='friend_02', nickname='friend_02', city='NYC', state='NY', age=30,
                interests=['film', 'art'], education='bachelor', industry='media',
                languages=['english']),
    UserProfile(id='friend_03', nickname='friend_03', city='Brooklyn', state='NY', age=26,
                interests=['music', 'film'], education='bachelor', industry='media',
                languages=['english']),
    UserProfile(id='friend_04', nickname='friend_04', city='NYC', state='NY', age=29,
                interests=['art', 'film', 'music'], education='master', industry='media',
                languages=['english', 'french']),
    UserProfile(id='friend_05', nickname='friend_05', city='Hoboken', state='NJ', age=31,
                interests=['art', 'music'], education='bachelor', industry='entertainment',
                languages=['english']),
]

# Interactions: Eleanor ↔ each friend (30 likes, 20 comments, 10 DMs)
FRIEND_INTERACTIONS = []
for f in FRIENDS:
    uid_a, uid_b = sorted(['eleanor', f.id])
    FRIEND_INTERACTIONS.append({
        'user_id_a': uid_a,
        'user_id_b': uid_b,
        'like_count': 30,
        'comment_count': 20,
        'dm_count': 10,
    })

# ---------------------------------------------------------------------------
# 94 padding users (varying profiles) — keeps n_samples > UMAP_N_NEIGHBORS=15
# ---------------------------------------------------------------------------
import random
random.seed(0)

_cities  = ['Chicago', 'LA', 'Seattle', 'Denver', 'Miami', 'Boston', 'Portland',
             'Atlanta', 'Dallas', 'Phoenix', 'Nashville', 'San Diego']
_states  = ['IL', 'CA', 'WA', 'CO', 'FL', 'MA', 'OR', 'GA', 'TX', 'AZ', 'TN', 'CA']
_interest_pool = ['gaming', 'hiking', 'travel', 'photography', 'cooking', 'fitness',
                  'reading', 'yoga', 'finance', 'sports', 'tech', 'fashion',
                  'gardening', 'DIY', 'movies', 'theatre', 'writing', 'science']
_education_pool = ['highschool', 'bachelor', 'master', 'phd', 'associate']
_industry_pool  = ['tech', 'finance', 'healthcare', 'education', 'retail',
                   'manufacturing', 'construction', 'legal', 'consulting']
_lang_pool = ['english', 'spanish', 'french', 'mandarin', 'arabic', 'portuguese',
               'japanese', 'korean', 'german', 'italian']

PADDING_USERS = []
for k in range(94):
    city_idx = k % len(_cities)
    PADDING_USERS.append(UserProfile(
        id=f'user_{k:03d}',
        nickname=f'user_{k:03d}',
        city=_cities[city_idx],
        state=_states[city_idx],
        age=random.randint(20, 55),
        interests=random.sample(_interest_pool, k=random.randint(1, 4)),
        education=random.choice(_education_pool),
        industry=random.choice(_industry_pool),
        languages=random.sample(_lang_pool, k=random.randint(1, 3)),
    ))

total = 2 + len(FRIENDS) + len(PADDING_USERS)
print(f'Core users: Eleanor + Brita + {len(FRIENDS)} friends + {len(PADDING_USERS)} padding = {total} total')

In [ ]:
# ---------------------------------------------------------------------------
# Plot helper
# ---------------------------------------------------------------------------

FRIEND_IDS = {f.id for f in FRIENDS}


def plot_frame(ax, profiles, interactions, title):
    """Run the full pipeline for one frame and plot on ax."""
    data = PipelineInput(profiles=profiles, interactions=interactions)
    dist_matrix = build_combined_distance(data)
    coords = project(dist_matrix)  # shape (N, 2)

    uid_to_idx = {p.id: i for i, p in enumerate(profiles)}
    eleanor_idx = uid_to_idx['eleanor']

    # Translate so Eleanor is at the origin
    coords = coords - coords[eleanor_idx]

    # Scatter: others first (grey), then friends (yellow), then Brita (red), Eleanor (blue)
    for i, p in enumerate(profiles):
        x, y = coords[i]
        if p.id == 'eleanor':
            ax.scatter(x, y, c='royalblue', marker='*', s=200, zorder=5, label='Eleanor')
        elif p.id == 'brita':
            ax.scatter(x, y, c='crimson', marker='o', s=80, zorder=4, label='Brita')
        elif p.id in FRIEND_IDS:
            ax.scatter(x, y, c='goldenrod', marker='^', s=60, zorder=3)
        else:
            ax.scatter(x, y, c='lightgrey', marker='.', s=20, zorder=1)

    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


print('plot_frame() defined')

In [ ]:
# ---------------------------------------------------------------------------
# Case 1 — Interaction convergence
# Brita's profile is fixed (maximally dissimilar to Eleanor).
# Eleanor↔Brita interaction intensity increases across 6 frames.
# ---------------------------------------------------------------------------

base_profiles = [ELEANOR, BRITA_ORIGINAL] + FRIENDS + PADDING_USERS

CASE1_FRAMES = [
    # (title, like_count, comment_count, dm_count)
    ('Frame 0: No interaction',           0,   0,  0),
    ('Frame 1: 5 likes',                  5,   0,  0),
    ('Frame 2: 20 likes',                20,   0,  0),
    ('Frame 3: 20 likes + 10 comments',  20,  10,  0),
    ('Frame 4: 20 likes + 30 comments',  20,  30,  0),
    ('Frame 5: 20 likes + 30 comments + 20 DMs', 20, 30, 20),
]

fig1, axes1 = plt.subplots(2, 3, figsize=(12, 8))
fig1.suptitle('Case 1 — Interaction Convergence (Brita profile fixed)', fontsize=12, fontweight='bold')

for ax, (title, likes, comments, dms) in zip(axes1.flat, CASE1_FRAMES):
    interactions = list(FRIEND_INTERACTIONS)  # copy
    if likes > 0 or comments > 0 or dms > 0:
        uid_a, uid_b = sorted(['eleanor', 'brita'])
        interactions.append({
            'user_id_a': uid_a,
            'user_id_b': uid_b,
            'like_count': likes,
            'comment_count': comments,
            'dm_count': dms,
        })
    plot_frame(ax, base_profiles, interactions, title)

# Add legend once
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='*', color='w', markerfacecolor='royalblue', markersize=12, label='Eleanor'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='crimson',   markersize=8,  label='Brita'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='goldenrod', markersize=8,  label='Friends'),
    Line2D([0], [0], marker='.', color='w', markerfacecolor='lightgrey', markersize=8,  label='Others'),
]
fig1.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, 0.01))

plt.tight_layout(rect=[0, 0.06, 1, 1])
os.makedirs('data', exist_ok=True)
fig1.savefig('data/case1.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved data/case1.png')

In [ ]:
# ---------------------------------------------------------------------------
# Case 2 — Profile convergence
# Eleanor↔Brita interaction = NONE throughout.
# Brita's profile changes per frame, converging toward Eleanor's.
# ---------------------------------------------------------------------------

def make_brita(**overrides):
    """Return a Brita UserProfile with selected fields overridden from BRITA_ORIGINAL."""
    base = BRITA_ORIGINAL.model_dump()
    base.update(overrides)
    return UserProfile(**base)


CASE2_BRITAS = [
    ('Frame 0: Original Brita',
     make_brita()),
    ('Frame 1: Matching interests',
     make_brita(interests=['art', 'music', 'film'])),
    ('Frame 2: + Same city/state',
     make_brita(interests=['art', 'music', 'film'], city='NYC', state='NY')),
    ('Frame 3: + Same age',
     make_brita(interests=['art', 'music', 'film'], city='NYC', state='NY', age=28)),
    ('Frame 4: + Same education',
     make_brita(interests=['art', 'music', 'film'], city='NYC', state='NY', age=28,
                education='bachelor')),
    ('Frame 5: Full profile match',
     make_brita(interests=['art', 'music', 'film'], city='NYC', state='NY', age=28,
                education='bachelor', industry='media', languages=['english'])),
]

fig2, axes2 = plt.subplots(2, 3, figsize=(12, 8))
fig2.suptitle('Case 2 — Profile Convergence (Eleanor↔Brita interaction = 0)', fontsize=12, fontweight='bold')

for ax, (title, brita_profile) in zip(axes2.flat, CASE2_BRITAS):
    profiles = [ELEANOR, brita_profile] + FRIENDS + PADDING_USERS
    plot_frame(ax, profiles, FRIEND_INTERACTIONS, title)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='*', color='w', markerfacecolor='royalblue', markersize=12, label='Eleanor'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='crimson',   markersize=8,  label='Brita'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='goldenrod', markersize=8,  label='Friends'),
    Line2D([0], [0], marker='.', color='w', markerfacecolor='lightgrey', markersize=8,  label='Others'),
]
fig2.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, 0.01))

plt.tight_layout(rect=[0, 0.06, 1, 1])
fig2.savefig('data/case2.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved data/case2.png')